#Initialization

In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import StringType, DateType
from pyspark.sql.window import Window
from pyspark.sql.functions import trim, col, length

#Read Bronze Table

In [0]:
df = spark.table("demo_project.bronze.crm_sales_details")

#Sanity checks

In [0]:
df.limit(10).display()

#Silver Transformations

##Renaming Columns

In [0]:
rename_map = {
    "sls_ord_num": "order_number",
    "sls_prd_key": "product_number",
    "sls_cust_id": "customer_id",
    "sls_order_dt": "order_date",
    "sls_ship_dt": "ship_date",
    "sls_due_dt": "due_date",
    "sls_sales": "sales_amount",
    "sls_quantity": "quantity",
    "sls_price": "price"
}

for old_name, new_name in rename_map.items():
    df = df.withColumnRenamed(old_name, new_name)

##Sanity checks

In [0]:
df.limit(10).display()

##Trimming

In [0]:
for field in df.schema.fields:
    if isinstance(field.dataType, StringType):
        df = df.withColumn(field.name, trim(col(field.name)))


##Cleaning Dates


In [0]:
from pyspark.sql.functions import col, length, trim, to_date, when

df = (
    df.withColumn(
        "order_date",
        when(
            ((col("order_date") == 0) | (length(col("order_date")) != 8)),
            None
        ).otherwise(to_date(col("order_date").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "ship_date",
        when(
            ((col("ship_date") == 0) | (length(col("ship_date")) != 8)),
            None
        ).otherwise(to_date(col("ship_date").cast("string"), "yyyyMMdd"))
    )
    .withColumn(
        "due_date",
        when(
            ((col("due_date") == 0) | (length(col("due_date")) != 8)),
            None
        ).otherwise(to_date(col("due_date").cast("string"), "yyyyMMdd"))
    )
)


##Sales and Price Corrections

In [0]:
df = (
    df
    .withColumn(
        "price",
        F.when(
            (col("price").isNull()) | (col("price") <= 0),
          F.when(
              col("quantity") != 0,
              col("sales_amount") / col("quantity")
            ).otherwise(None)
          ).otherwise(col("price"))

    )
)

##Sanity checks

In [0]:
df.limit(10).display()

#Writing Silver Table

In [0]:
df.write.mode("overwrite").format("delta").saveAsTable("demo_project.silver.crm_sales")

##Sanity checks of Silver Table

In [0]:
%sql
select * from demo_project.silver.crm_sales 
limit 10;